In [ ]:
import requests
from urllib.parse import unquote
import re
import pygsheets
import pandas as pd

In [ ]:
quantidade_extrair = 25

In [ ]:
client = pygsheets.authorize(service_account_file=r'C:\Users\fumio\Documents\gcp-project-365616-6b0f83cca4c2.json')

ss = client.open_by_url('https://docs.google.com/spreadsheets/d/1H-otjPW382f7cUNvaC_p5BJCMw7nmlmuhu2JiNBx_Y0/edit?gid=195125089#gid=195125089')

In [ ]:
ws = ss.worksheet_by_title('Nomes Para SeekLoc')

In [ ]:
df = ws.get_as_df()

In [ ]:
len(df)

In [ ]:
df_filtrado = df[df['Telefone'] == '']
df_filtrado = df_filtrado[df_filtrado['Whatsapp'] == '']
df_filtrado = df_filtrado[df_filtrado['Proprietário'] != '']
df_filtrado = df_filtrado[df_filtrado['Anotações Automação'].str.contains(' - ')]
df_filtrado = df_filtrado[df_filtrado['Endereço'].str.contains('AP 15')]
df_filtrado = df_filtrado[df_filtrado['Contato Feito'] == 'FALSE']
# df_filtrado = df_filtrado[df_filtrado['Quantidade de Imóveis'] > 3]

In [ ]:
filtros = [
    'Condomínio Ext Praça Morumbi',
    # 'Cristais da Terra - Panamby',
    # 'Reserva Morumbi',
    # 'Terras da Mata',
    # 'Mais Morumbi Clube',
    # 'Passarim',
    # 'Antigua Morumbi',
    # 'Condomínio Luiza',
]

In [ ]:
df_filtrado = df_filtrado[df_filtrado['Endereço'].apply(lambda x: any([filtro in x for filtro in filtros]))]

In [ ]:
# df_filtrado = df_filtrado.sort_values('Quantidade de Imóveis', ascending=False)

In [ ]:
len(df_filtrado)

In [ ]:
df_filtrado = df_filtrado[:quantidade_extrair]

In [ ]:
len(df_filtrado)

In [ ]:
df_filtrado

In [ ]:
headers = {
    "accept": "text/javascript, text/html, application/xml, text/xml, */*",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "x-prototype-version": "1.6.0",
    "x-requested-with": "XMLHttpRequest",
    "user-agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "referrer": "http://200.201.193.100/seekloc/sistema.php",
    "cookie" : "supremeseekloc=c9jt2o3opbbfn9crcj1ve0re64",
    "referrerPolicy": "strict-origin-when-cross-origin",
  }

In [ ]:
def extrair_contatos(nome_cpf) -> tuple:
    if ' - ' not in nome_cpf:
        return 'Sem informação necessária.', 'Sem informação necessária.'

    cpf, get_dados = nome_cpf.split(' - ')

    params = {
        "action" : "getdados",
        "id" : get_dados,
    }

    try:
        response = requests.get(
            url='http://200.201.193.100/seekloc/ajax.php',
            params = params,
            headers = headers,
        )
    
    except Exception as e:
        return f"Erro requests: {e}", f"Erro requests: {e}"
    
    try:
        response_text = unquote(response.content.decode(encoding='latin-1'))

        telefones = re.findall(r'getdadostel\(\d+\).>(\(\d{2}\) [0-8][\d-]+)</span>', response_text)
        celulares = re.findall(r'getdadostel\(\d+\).>(\(\d{2}\) 9[\d-]+)</span>', response_text)

        output_telefone = 'Não encontrado.' if len(telefones) == 0 else '\n'.join(telefones) 
        output_celular = 'Não encontrado.' if len(celulares) == 0 else '\n'.join(celulares) 

        return output_telefone, output_celular
    
    except Exception as e:
        return f"Erro processamento: {e}", f"Erro processamento: {e}"


In [ ]:
df_filtrado[['Telefone', 'Whatsapp']] = df_filtrado['Anotações Automação'].apply(lambda x: pd.Series(extrair_contatos(x)))

In [ ]:
ws = ss.worksheet_by_title('Nomes Para SeekLoc')
df = ws.get_as_df()

In [ ]:
df.update(df_filtrado)

In [ ]:
ws.set_dataframe(
    df,
    'A1'
)